In [1]:
import chromadb
chroma_client = chromadb.Client()

In [2]:
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

sentence_transformer_ef = SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L12-v2",
    device="cuda",
    normalize_embeddings=True
)

print("SENTENCE TRANSFORMER")

/run/media/tahas44/Yeni Birim/Technarts/Intern/NLP/InformationRetrieval/.venv/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2861.56it/s]


SENTENCE TRANSFORMER


In [3]:
collection = chroma_client.get_or_create_collection(
    name="IR_minilm_l12",
    embedding_function=sentence_transformer_ef
)

In [4]:
import ir_datasets
dataset = ir_datasets.load("wikir/en1k/training")
print("Data set yüklendi")

Data set yüklendi


In [5]:
doc_texts = []
doc_ids = []

for doc in dataset.docs_iter():
    doc_texts.append(doc.text)
    doc_ids.append(doc.doc_id)

### Vector Database Storage

In [6]:
from tqdm import tqdm

BATCH_SIZE = 5000
total_doc = len(doc_texts)

for i in tqdm(range(0, total_doc, BATCH_SIZE)):
    batch_texts = doc_texts[i: i + BATCH_SIZE]
    batch_ids = doc_ids[i: i + BATCH_SIZE]

    collection.add(
        documents=batch_texts,
        ids=batch_ids
    )

print("Vektorization completed!")

100%|██████████| 74/74 [1:05:51<00:00, 53.40s/it]

Vektorization completed!


In [7]:
queries = []

for query in dataset.queries_iter():
    queries.append(query.text)

results_10 = collection.query(
    query_texts=queries,
    n_results=10
)

print("En iyi 10 doküman!")

results_5 = collection.query(
    query_texts=queries,
    n_results=5
)

print("En iyi 5 doküman!")

from collections import defaultdict

qrels_dict = defaultdict(list)

for qrel in dataset.qrels_iter():
    qrels_dict[qrel.query_id].append(qrel.doc_id)

qrels_dict = dict(qrels_dict)

query_ids = [query.query_id for query in dataset.queries_iter()]

class Scoredoc:
    def __init__(self, doc_id, score):
        self.doc_id = doc_id
        self.score = score

score_doc_dict = defaultdict(list)

for scoreddoc in dataset.scoreddocs_iter():
    doc_id = scoreddoc.doc_id
    score = scoreddoc.score

    scoreddoc_object = Scoredoc(doc_id, score)

    score_doc_dict[scoreddoc.query_id].append(scoreddoc_object)

En iyi 10 doküman!
En iyi 5 doküman!


In [8]:
from helper import pipeline
df_parquet = pipeline(results_5, results_10, query_ids, "SenTransformer minilm_l12", qrels_dict, score_doc_dict)

     Query_ID   recall_5  precision_5      AP_5    NDCG_5  recall_10  \
0      123839  16.666667         20.0  0.041667  0.430677  33.333333   
1      188629  16.666667         20.0  0.166667  1.000000  16.666667   
2       13898  33.333333         40.0  0.333333  0.000000  33.333333   
3      316959  22.222222         40.0  0.185185  0.912697  22.222222   
4      515031   7.142857         20.0  0.071429  0.498714   7.142857   
...       ...        ...          ...       ...       ...        ...   
1439   896124   0.000000          0.0  0.000000  0.671522  12.500000   
1440    12319   4.545455         20.0  0.022727  0.994133   4.545455   
1441     4421   0.000000          0.0  0.000000  0.707357   0.000000   
1442   296526  10.000000         20.0  0.000000  0.701904  20.000000   
1443   341793  14.285714         20.0  0.142857  0.880847  14.285714   

      precision_10     AP_10   NDCG_10  f_score_5  f_score_10  
0             20.0  0.089286  0.460969  18.181818   25.000000  
1      

In [9]:
df_parquet

,Method,recall_5_mean,recall_5_std,recall_5_max,recall_5_min,recall_10_mean,recall_10_std,recall_10_max,recall_10_min,precision_5_mean,...,MAP_5,MAP_10,NDCG_5_mean,NDCG_5_std,NDCG_5_max,NDCG_5_min,NDCG_10_mean,NDCG_10_std,NDCG_10_max,NDCG_10_min
0,SenTransformer minilm_l12,12.984136,12.349759,83.333333,0.0,16.480406,15.229594,100.0,0.0,28.573407,...,0.099883,0.118783,0.708287,0.353963,1.0,0.0,0.7091,0.296196,1.0,0.0


In [10]:
df_parquet.to_parquet("SenTransformerMinilm_l12.parquet")